# Sizing & tuning

You have two levers for shrinking the data: the number of **periods** and the
number of **segments** per period (see [Segmentation](segmentation.ipynb)). This
page is about choosing them — first by hand, then by letting tsam search.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## The accuracy–size trade-off

More time steps means more accuracy but less compression. Sweeping the number of
periods shows the classic diminishing-returns curve — accuracy falls fast, then
flattens:

In [ ]:
rows = []
for k in [2, 4, 6, 8, 12, 16]:
    r = tsam.aggregate(data, n_clusters=k, period_duration="1D")
    rows.append(
        {
            "n_clusters": k,
            "timesteps": k * r.n_timesteps_per_period,
            "rmse": float(r.accuracy.rmse.mean()),
        }
    )
sweep = pd.DataFrame(rows)
px.line(
    sweep,
    x="timesteps",
    y="rmse",
    markers=True,
    title="Accuracy vs. size — number of periods",
)

## Let tsam choose

Both levers interact, so searching them by hand is tedious. Give
[`find_optimal_combination`](../api/tuning.md) a target **data reduction** (here,
keep 10% of the time steps) and it searches period/segment combinations for the
most accurate one that hits the target:

In [ ]:
from tsam.tuning import find_optimal_combination

best = find_optimal_combination(
    data,
    data_reduction=0.1,
    period_duration="1D",
    n_jobs=1,
    show_progress=False,
)
print(
    f"{best.n_clusters} periods x {best.n_segments} segments  ->  RMSE {best.rmse:.4f}"
)

## The whole frontier

To choose deliberately rather than against a fixed target,
[`find_pareto_front`](../api/tuning.md) maps the accuracy-vs-size frontier — every
point is the best aggregation for that many time steps:

In [ ]:
from tsam.tuning import find_pareto_front

pareto = find_pareto_front(
    data,
    period_duration="1D",
    timesteps=range(12, 84, 12),
    n_jobs=1,
    show_progress=False,
)
pareto.summary

In [ ]:
pareto.plot()

`best.best_result` — and every point on the Pareto front — is an ordinary
aggregation result, ready to hand to a model. See the
[Optimization workflow](optimization_workflow.ipynb).